# Case Studies

## What's covered

Five canonical system design problems, worked end-to-end. Each starts from requirements and capacity estimates, sketches the data model and API, walks the request path through the architecture, and calls out the trade-offs.

1. **URL shortener** — read-heavy, write-light, latency-sensitive
2. **Rate limiter** — high-throughput, distributed-state, accuracy/latency trade-off
3. **Twitter-style feed** — fan-out on write vs read, the celebrity-tweet problem
4. **Chat system** — real-time, presence, message delivery guarantees
5. **Distributed cache** — consistent hashing, replication, hot keys

The vocabulary from notebooks one through seven is the design language for all five.


## How to read these designs

Each case study follows the same four-step structure that's worth using on any system-design question, in interviews or in practice:

1. **Requirements.** What does the system have to do, and what does it have to *not* do? Functional requirements (the features) and non-functional requirements (scale, latency, consistency, availability).
2. **Capacity estimates.** Run the back-of-envelope math from notebook 01 — daily users, QPS, storage growth, bandwidth. The numbers narrow the design space.
3. **High-level architecture.** Sketch the components, data flow, data store choices, and request path. This is the "draw boxes and arrows" part.
4. **Trade-offs and scale.** Where will this hit a wall? What did we deliberately give up? How does the design evolve at 10x scale?

Skipping any of these four — especially the numbers — is the classic mistake. The numbers are how you justify every choice that follows.


## Case 1 — URL shortener

**The problem.** Take a long URL like `https://example.com/very/long/path?query=...` and return a short URL like `https://sho.rt/aB3xK9`. When a user visits the short URL, redirect them to the original.

### Requirements

**Functional:**
- Submit a long URL, get a short URL back.
- GET on the short URL redirects (301 or 302) to the long URL.
- Optionally, custom aliases (`sho.rt/my-name`) and expiry dates.

**Non-functional:**
- Reads vastly outnumber writes (~1000:1 typical for link-shortening services).
- Redirect latency must be very low (every user is waiting on it). Target p99 < 50ms end-to-end.
- High availability — a broken shortener breaks every link anyone has shared.
- Links don't change after creation; they just exist or don't.

### Capacity estimates

Suppose 100M new links per month, 100B clicks per month.

- **Writes:** 100M / 30 / 86,400 ≈ 40 writes/sec. Tiny.
- **Reads:** 100B / 30 / 86,400 ≈ 40,000 reads/sec. Significant.
- **Storage per link:** ~500 bytes (long URL, short code, owner, created_at, expires_at, click count).
- **Storage growth:** 100M × 500B = 50 GB/month, 600 GB/year. Easy.
- **Bandwidth:** 40K reads/sec × 500B response ≈ 20 MB/sec on redirect responses.

The numbers tell us this is fundamentally a **read-heavy, low-latency lookup problem**. Storage volume is small. Compute per request is minimal. The whole game is "given a 6-char code, return the long URL fast."

### API
```
POST /shorten   { url, custom_alias?, expires_at? } -> { short_url, short_code }
GET  /:code     -> 302 redirect to long URL
```

### Architecture
```
   browser
      |
      v
   CDN (caches popular redirects at edge with short TTL)
      |
      v
   load balancer
      |
      v
   stateless web tier (Node/Go/Python — many instances)
      |
      | --- on read miss, look up in cache
      v
   Redis cache (short_code -> long_url)
      |
      | --- on cache miss, look up in DB
      v
   database (DynamoDB / Cassandra / sharded MySQL by short_code)
```

### Generating short codes

The simplest scheme: assign a monotonically increasing integer ID, base62-encode it (using `[a-zA-Z0-9]` = 62 characters). A 6-character base62 code addresses 62^6 ≈ 56 billion URLs — enough for years. We'll code this in a moment.

**Two alternatives:**

- **Random 6-char codes.** Generate, check for collision, retry. Simpler, but collision probability grows with the database. At 50 billion URLs, every new code has a noticeable collision rate.
- **Hash the long URL.** Take SHA-256, truncate to 6 base62 chars. Deterministic (same URL → same code), but you have to handle the case where two URLs hash to the same code, and you can't shorten the same URL into different codes.

Sequential base62 is the dominant choice. The trade-off is that codes are guessable — bots can crawl `aB3xK0, aB3xK1, ...` to discover links. For most public-link-shortener use cases this is acceptable; for private links, mix in randomness.

### Trade-offs and scale

- **The cache hit rate is everything.** Most clicks are on a small set of popular links; a small in-memory cache catches the vast majority. Cache miss penalty is one DB lookup (~2ms); cache hit is ~0.1ms.
- **Read replicas of the database** handle the 40K reads/sec without straining the primary.
- **Asynchronous click counting.** Don't update a counter in the DB on every click — emit a click event to a queue, aggregate downstream. Otherwise the click-update write rate dominates the link-creation write rate.
- **CDN at the edge** can serve redirects from the closest POP, which drops global users' latency dramatically. Set a short TTL (5-60 minutes) so revoked links propagate within that window.


### Base62 encoder — the core piece


In [ ]:
# Base62 encoding for URL-shortener IDs.
# 62 characters: 0-9, a-z, A-Z. A 6-char code addresses 62^6 ~= 56 billion URLs.

ALPHABET = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
BASE = len(ALPHABET)


def encode(n: int) -> str:
    if n == 0:
        return ALPHABET[0]
    digits = []
    while n > 0:
        digits.append(ALPHABET[n % BASE])
        n //= BASE
    return "".join(reversed(digits))


def decode(s: str) -> int:
    n = 0
    for ch in s:
        n = n * BASE + ALPHABET.index(ch)
    return n


# Demonstrate the address space.
print(f"id     1 -> code {encode(1)!r}")
print(f"id    62 -> code {encode(62)!r}")
print(f"id   100 -> code {encode(100)!r}")
print(f"id 1e9   -> code {encode(10**9)!r}     ({len(encode(10**9))} chars)")
print(f"id 1e12  -> code {encode(10**12)!r}    ({len(encode(10**12))} chars)")

# Round-trip check.
for original in [42, 1_000_000, 56_800_235_584]:
    code_str = encode(original)
    back = decode(code_str)
    print(f"  {original} -> {code_str!r} -> {back}   {'ok' if back == original else 'MISMATCH'}")

# Address space.
print(f"\n6-char codes address {BASE**6:,} URLs")
print(f"7-char codes address {BASE**7:,} URLs")


## Case 2 — Distributed rate limiter

**The problem.** Allow each API key to make at most N requests per minute. Reject the rest with HTTP 429. The limit is enforced across a fleet of servers in different regions.

### Requirements

**Functional:**
- Per-API-key limits (e.g. 100 req/min).
- Configurable limits per tier (free, pro, enterprise).
- Return rate-limit headers (`X-RateLimit-Limit`, `X-RateLimit-Remaining`, `X-RateLimit-Reset`).

**Non-functional:**
- Decision must be fast (sub-millisecond at p99).
- Accurate within a few percent — exact counts not required.
- Highly available — limiter failure must not block all traffic.

### Capacity estimates

10M API keys, 10K requests/sec across the fleet, 50 servers behind the limiter.

- **Decisions per second:** 10K (one per incoming request).
- **State per key:** ~50 bytes (current count, window start, last seen).
- **Total state:** 10M × 50B = 500 MB. Fits comfortably in Redis on one machine.
- **Hot keys:** the top 1% of API keys generate ~80% of traffic. Sharding by key handles this.

### Architecture

The choice from notebook 02: token-bucket, centralized counter, or two-tier approximation.

```
                       per-server                   centralized
                      token bucket                    counter
                          v                              v
                  +-------------+              +-------------------+
   request --->   | app server  |---ask once->| Redis cluster     |
                  +-------------+    every Ns  | sharded by API key|
                  | local count |               +-------------------+
                  +-------------+                       ^
                                                        |
                  shares back rate-limit counts every N seconds
```

**The two-tier design.** Each application server has a *local* approximate counter. On every Nth request (or every T milliseconds), the server syncs with the central Redis cluster — reports its local count, gets back the latest authoritative count. Between syncs, the local counter is the source of truth. The trade-off is that the limit may be exceeded by up to (servers × sync_interval × rate) at the edges — typically a few percent over.

This buys sub-millisecond decisions (local lookup) with global enforcement (central reconciliation). The pure centralized approach would add a Redis round-trip to every request — fine at modest scale, painful at 10K QPS.

### Trade-offs and scale

- **Failure mode.** If Redis is down, options are: fail open (allow everything — limits don't matter while Redis recovers) or fail closed (reject everything — safer for the protected service). Most production limiters fail open.
- **Window choice.** Token bucket (from notebook 02) is the typical implementation — refill rate + burst capacity. Avoids both fixed-window boundary issues and sliding-window memory cost.
- **Sharding.** Redis-cluster shards by API key. Hot keys can be replicated or pinned to dedicated shards.
- **Client side.** Return `Retry-After` so well-behaved clients back off properly. Don't just reject silently — that prompts blind retries that make it worse.


## Case 3 — Twitter-style timeline

**The problem.** A user has a timeline of recent posts from people they follow. New posts appear at the top in roughly chronological order. The user pulls down the feed and reads it.

### Requirements

**Functional:**
- Post (tweet/status/note) by a user.
- Follow another user.
- View timeline: most recent posts from followed users.

**Non-functional:**
- Read-heavy. Most users read their feed many times per day; few post.
- Reads must be very fast (the feed is the main UX).
- Eventual consistency is fine — a new post not appearing for a few seconds is acceptable.
- Some users have hundreds of millions of followers (the celebrity-tweet problem).

### Capacity estimates

300M monthly users, 150M daily active. Average user follows 200, posts 0.5/day, reads feed 20×/day.

- **Posts per second:** 150M × 0.5 / 86,400 ≈ 870/sec. Peak ~3× = 2,500/sec.
- **Timeline reads per second:** 150M × 20 / 86,400 ≈ 35,000/sec. Peak ~100,000/sec.
- **Storage per post:** ~500 bytes core + media references = 1 KB.
- **Storage growth:** 75M posts/day × 1KB = 75 GB/day = 27 TB/year. Significant but manageable.

The read/write ratio is roughly 40:1. The architecture has to be optimized for reads.

### Two architectural choices: fan-out on write vs fan-out on read

**Fan-out on write (push model).** When a user posts, write the post ID into the timeline of every follower. Reading a timeline is just `SELECT * FROM timeline WHERE user_id = X ORDER BY ts DESC LIMIT 50` — a cheap lookup.

- **Pros:** reads are O(1) on a denormalized list.
- **Cons:** every post becomes N writes (N = follower count). For a celebrity with 100M followers, one post is 100M writes — too expensive.

**Fan-out on read (pull model).** When a user reads their timeline, query the posts of every user they follow and merge by timestamp. Writes are O(1); reads are O(follow count).

- **Pros:** writes are cheap, easy.
- **Cons:** reads are expensive — a user following 1,000 people triggers 1,000 lookups + merge. At scale, this dwarfs the write cost.

**The hybrid.** Most production systems fan out on write for *normal* users, and fan out on read for *celebrities*. A user's timeline is the merge of (a) their precomputed timeline of normal-user posts and (b) freshly fetched recent posts from celebrity follows. Celebrity is defined by threshold (e.g. follower count > 1M).

### Architecture
```
   post                    timeline read
   ----                    -------------
                                   |
   user posts                      v
        |                  timeline service
        v                          |
   post service                    | check follower list
        |                          v
        | publish to fan-out queue
        v
   fan-out worker
        |
        v
   for each follower:
     push to follower's timeline cache (Redis sorted set by ts)
        |
   (celebrities skipped — fetched on read instead)
```

### Trade-offs and scale

- **Storage of timelines.** Each user's timeline is the last ~1000 post IDs in a Redis sorted set. 300M users × 1000 entries × ~20 bytes = ~6 TB across the cluster. Fits.
- **Cold-start.** A user who hasn't logged in for a month may have an empty cached timeline. Rebuild it on read by querying their follow graph and merging.
- **Real-time updates.** Long-polling, WebSocket, or Server-Sent Events from the timeline service push new posts as they arrive — covered next in the chat system.
- **Search and trending** are separate systems (Elasticsearch for search; stream aggregation for trends) — out of scope for the timeline itself.


## Case 4 — Chat system

**The problem.** Users exchange messages in 1:1 and group conversations. Messages are delivered in near-real-time. Presence (online/offline) is shown. History is preserved.

### Requirements

**Functional:**
- Send a message in a 1:1 or group conversation.
- Receive messages in real-time when online.
- Presence indicators (online, last-seen).
- Message history.
- Read receipts and delivery confirmations.

**Non-functional:**
- Low end-to-end latency (< 200ms in most cases).
- Reliable delivery — messages must not be lost.
- Scalable to millions of concurrent connections.

### Capacity estimates

500M users, 100M daily active, average 50 messages/user/day.

- **Messages per second:** 100M × 50 / 86,400 ≈ 58,000/sec. Peak ~3× = ~175K/sec.
- **Concurrent connections:** assume 50% of DAU are online concurrently at peak = 50M concurrent WebSocket connections.
- **Storage per message:** ~250 bytes core + attachments by reference = 500 bytes.
- **Storage growth:** 5B messages/day × 500B = 2.5 TB/day = ~900 TB/year. Large, but sharded by conversation.

### Architecture
```
   client (WebSocket)
        |
        v
   load balancer (sticky by connection)
        |
        v
   chat server (holds long-lived WS connections)
        |
        | message in
        v
   publish to message bus (Kafka / Redis Streams)
        |
        v
   ┌────────────┬────────────────┬────────────┐
   v            v                v            v
 message      delivery        notification   presence
 store        service         service        service
 (Cassandra)  (recipients'    (push if       (Redis)
              chat servers)   offline)
```

### The connection-routing problem

50M concurrent connections doesn't fit on one server. Connections are distributed across thousands of chat servers. When user A sends to user B:

1. A's chat server publishes the message to the message bus.
2. The delivery service knows which chat server B is connected to (via a Redis directory: `user_id → server_id`).
3. The delivery service sends the message to B's chat server.
4. B's chat server writes to B's WebSocket.

If B is offline, the message is stored in B's inbox in the message store; on next connection, B receives the backlog. A push notification (APNS/FCM) wakes up B's mobile app.

### Trade-offs and scale

- **WebSocket vs long-polling.** WebSocket is the default for real-time. Long-polling is the fallback when WebSocket isn't available (some corporate networks). Server-Sent Events is a one-way alternative useful for notification-only feeds.
- **Message ordering.** Per-conversation ordering only. Across conversations, events can interleave. Per-conversation messages have a monotonically increasing sequence number to detect gaps.
- **Reliability.** At-least-once delivery with deduplication on the client (by message ID). Same pattern as notebook 05.
- **Group chats.** A group message fans out to N recipients. For small groups (< 1000), direct fan-out. For large channels (Slack-style), the server publishes once and clients subscribe; presence is computed differently because not everyone is "in" the channel at once.
- **End-to-end encryption** (Signal, iMessage, WhatsApp) is layered on top. Messages are ciphertext from the server's perspective; the server only sees routing metadata. Complicates server-side features (search, ad serving) but is a separate problem.


## Case 5 — Distributed cache

**The problem.** Build a system where applications store and retrieve cached values across a cluster. Sub-millisecond reads, sharded across many machines, replicated for fault tolerance.

### Requirements

**Functional:**
- `GET(key)` returns the cached value or null.
- `SET(key, value, ttl)` stores a value, optionally with an expiry.
- `DELETE(key)` evicts.
- Capacity in the terabytes.

**Non-functional:**
- p99 read latency < 1ms.
- Highly available — single machine failure doesn't take a key offline.
- Linear scale — adding capacity adds throughput proportionally.

### Capacity estimates

5 TB of hot data, 1M reads/sec, 100K writes/sec, 1 KB average value size.

- **RAM per node:** 256 GB → need at least 20 nodes for raw storage, more for replication.
- **Reads per node:** 1M / 20 = 50K reads/sec per node. Comfortable.
- **Replication:** 2-3 replicas per key. So actual cluster size = 40-60 nodes.

### Architecture — built on patterns from earlier notebooks

```
   client  --(consistent hashing on key)-->  picks shard
                                                |
                                                v
                                       primary cache node
                                                |
                                                | async to replicas
                                                v
                                       replica 1, replica 2
```

**Consistent hashing** (notebook 02) maps each key to a shard. Each shard has a primary plus 1-2 replicas. Reads go to the primary by default; replicas serve reads when the primary is down. Writes go to the primary; the primary replicates asynchronously.

**Why async replication?** The cache is *for* low latency. Synchronous replication would multiply write latency by the slowest replica's round-trip. The trade-off is that a primary failure might lose the most recent writes (cache-as-source-of-truth scenarios need a different architecture).

### The hot-key problem

If one key gets 10% of all traffic — a celebrity's profile, today's trending topic — its shard is overwhelmed even when the cluster is otherwise fine.

**Solutions:**

- **Per-key replication.** Detect hot keys and replicate them across multiple nodes. Reads spread; writes still go to the primary.
- **In-process client-side cache.** Each app server has its own LRU cache for the very hottest keys. Cuts the distributed-cache load by an order of magnitude for those keys.
- **Probabilistic refresh.** When a TTL is about to expire, only one client refreshes — others use the still-cached value until refreshed. Prevents the "thundering herd" on expiry.

### Cluster resizing

When the cluster grows from 20 to 25 nodes:

- **Consistent hashing** ensures only 1/5 of keys move.
- During migration, the new node *fetches keys on demand* from the old owner (lazy migration) or pre-warms by reading the whole old-owner shard (eager migration). Lazy is simpler; eager is faster.
- The client's hash ring is updated atomically when the new node is ready.

### Trade-offs and scale

- **Eviction policy.** LRU is the default. LFU for skewed access patterns. TTL-based eviction is independent of LRU/LFU and runs in parallel.
- **Persistence.** Cache nodes typically don't persist — the data is recomputable from the source of truth. Redis has optional AOF/RDB persistence for cache-as-data-store use cases.
- **Multi-region.** Cross-region replication is async only, with conflict resolution by last-write-wins or per-key vector clocks (notebook 04 vocabulary). Strong consistency across regions defeats the latency win — pick the right tool.

The dominant production implementations are **Redis Cluster** (single-master per shard, async replicas), **Memcached + consistent hashing on the client**, and **proprietary caches inside Dynamo/Cassandra/etc.**


## The throughline

Eight notebooks of system design vocabulary, and the five case studies above are where the vocabulary becomes the design language. Look back at the five designs and notice how much of the *same toolkit* appears:

- **Consistent hashing** in the rate limiter, the distributed cache, the chat connection directory.
- **At-least-once + idempotent processing** in the rate limiter sync, the message delivery, the click counter.
- **Cache-aside** in the URL shortener and as the read path in every other design.
- **Async replication** in the cache, the timeline cache, the message store.
- **Fan-out on write vs fan-out on read** explicit in the timeline, implicit in chat group delivery.
- **Eventual consistency** baked into all five — strong consistency was never the right choice for any of them.
- **The latency table from notebook 01** is, in the end, what drove every architectural choice — caching exists because RAM is 1000× faster than SSD, CDNs exist because cross-continent round-trips are 100× longer than same-DC, and so on.

The throughline of the series, in one sentence: **system design is the discipline of trading the right things for the right things at the right scale.** No design is a recipe. Every design is a set of decisions, each justified by numbers and constrained by the laws of distributed systems. The vocabulary lets the trade-offs be discussed; the numbers make them defensible; the operational practices keep them honest.

That's the series. Build well, measure honestly, fail gracefully.
